# Chapter 24 — The Models Were Different. Their Mistakes Weren't

**Companion to Applied AI**

Question: Does nominal model diversity produce effective diversity?

By the end of this notebook you will have:

- simulated five independent agents vs five sharing a latent failure mode
- measured pairwise error overlap (correlation)
- shown shared flaws erasing the portfolio's edge

## What this notebook demonstrates
A correlated-error simulation of the *mechanism*: shared latent flaws collapse effective diversity. Agents, tasks, and rates are synthetic.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

seed: 42


## 1. Two portfolios, same marginal skill

In [2]:
N_TASKS, N_AGENTS, P_ERR = 60, 5, 0.25
def run_portfolio(shared_flaw: bool):
    rng = random.Random(SEED)
    flaw_tasks = set(rng.sample(range(N_TASKS), 12)) if shared_flaw else set()
    errs = []
    for _ in range(N_AGENTS):
        e = set()
        for t in range(N_TASKS):
            if t in flaw_tasks and rng.random() < 0.9:
                e.add(t)
            elif rng.random() < P_ERR:
                e.add(t)
        errs.append(e)
    return errs

ind_errors = run_portfolio(False)
shr_errors = run_portfolio(True)
print("mean errors/agent independent:", sum(map(len, ind_errors)) / N_AGENTS)
print("mean errors/agent shared-flaw:", sum(map(len, shr_errors)) / N_AGENTS)

mean errors/agent independent: 16.8
mean errors/agent shared-flaw: 23.6


## 2. Pairwise error overlap: the diversity you actually got

In [3]:
def mean_overlap(errs) -> float:
    pairs = [(a, b) for i, a in enumerate(errs) for b in errs[i + 1:]]
    return sum(len(a & b) / max(1, len(a | b)) for a, b in pairs) / len(pairs)

oi, os = mean_overlap(ind_errors), mean_overlap(shr_errors)
print(f"mean pairwise error overlap: independent={oi:.2f}  shared-flaw={os:.2f}")
assert os > oi

mean pairwise error overlap: independent=0.17  shared-flaw=0.41


## 3. Oracle coverage: does any agent solve each task?

In [4]:
def oracle_cover(errs) -> float:
    all_err = set.intersection(*errs) if errs else set()
    return 1 - len(all_err) / N_TASKS

print(f"tasks solved by >=1 agent: independent={oracle_cover(ind_errors):.2f}  shared={oracle_cover(shr_errors):.2f}")
assert oracle_cover(shr_errors) <= oracle_cover(ind_errors)

tasks solved by >=1 agent: independent=1.00  shared=0.85


## Interpretation
- Supports: nominal diversity (five agents) can carry near-zero effective diversity when failures correlate; measure overlap, don't assume it.
- Does NOT support: claims about any real model family.

## Try it yourself
1. Vary the flaw-task count from 0 to 30 and plot overlap.
2. Compute a rescue set: tasks the portfolio solves that the best single agent misses.
3. Add a sixth agent with a *different* flaw set and watch coverage recover.